In [1]:
# importando os scripts de consulta

import sys
import os

# Altera o caminho para a pasta raiz do projeto (um nível acima)
raiz_projeto = os.path.abspath(os.path.join(os.getcwd(), ".."))

# Adiciona a raiz do projeto ao caminho de busca do Python se ainda não estiver lá
if raiz_projeto not in sys.path:
    sys.path.append(raiz_projeto)

# Agora você pode importar os módulos da pasta scripts usando a notação de ponto
from scripts.crm_leads import load_leads
from scripts.crm_inscricoes import load_inscricoes





In [2]:
leads = load_leads()


Página 1: 250 registros
Página 2: 250 registros
Página 3: 250 registros
Página 4: 250 registros
Página 5: 250 registros
Página 6: 250 registros
Página 7: 250 registros
Página 8: 250 registros
Página 9: 250 registros
Página 10: 250 registros
Página 11: 250 registros
Página 12: 250 registros
Página 13: 250 registros
Página 14: 250 registros
Página 15: 250 registros
Página 16: 250 registros
Página 17: 250 registros
Página 18: 250 registros
Página 19: 250 registros
Página 20: 250 registros
Página 21: 250 registros
Página 22: 250 registros
Página 23: 250 registros
Página 24: 250 registros
Página 25: 250 registros
Página 26: 250 registros
Página 27: 250 registros
Página 28: 250 registros
Página 29: 250 registros
Página 30: 250 registros
Página 31: 250 registros
Página 32: 250 registros
Página 33: 250 registros
Página 34: 250 registros
Página 35: 250 registros
Página 36: 250 registros
Página 37: 250 registros
Página 38: 250 registros
Página 39: 250 registros
Página 40: 250 registros
Página 41

In [4]:
"""
CRM Educacional -> Power BI
Script refatorado (template)

Preencha a lista `atributos` e o dicionário `rename_columns`
com os valores do seu projeto.
"""

import time
import requests
import xmltodict
import pandas as pd

BASE_URL = "https://trivento.api.crmeducacional.com"
URL = f"{BASE_URL}/api/LocalizacaoAvancada"

USERNAME = "IntegraçãoCRM"
PASSWORD = "IntegraçãoCRM"
AUTH_TYPE = "APICRMEducacional"

atributos = [
    'cad_numeroinscricao',
    "cad_clientepotencialid",
    "crmeduc_data_aprovacao",
    "cad_datapagamento",
    "cad_situacaoavaliacao",
    "cad_situacao",
    "cad_ofertaid",
    "trivento_unidadedeensinotrivento",
    "cad_concursoid",
    "coc_situacaopagamento",
]

rename_columns = {
    'cad_inscricaocandidatoid': 'id_inscricao',
    "cad_clientepotencialid": "ID Lead",
    'cad_numeroinscricao': 'Numero Inscricao',
    "crmeduc_data_aprovacao": "Data Aprovação",
    "cad_datapagamento": "Data de Pagamento",
    "cad_situacaoavaliacao": "Situação de Avaliação",
    "cad_situacao": "Situação Inscrição",
    "cad_ofertaid": "Curso ID",
    "trivento_unidadedeensinotrivento": "Unidade Inscrito",
    "cad_concursoid": "Forma de Ingresso",
    "coc_situacaopagamento": "Situação Pagamento",
}


def get_token(session):
    payload = {
        "grant_type": "password",
        "username": USERNAME,
        "password": PASSWORD,
        "authentication_type": AUTH_TYPE,
    }

    response = session.post(
        f"{BASE_URL}/api/token",
        data=payload,
        headers={"Content-Type": "application/x-www-form-urlencoded"},
        timeout=30,
    )
    response.raise_for_status()
    return response.json()["access_token"]


def create_headers(token):
    return {
        "Authorization": f"Bearer {token}",
        "Content-Type": "application/xml",
    }


def create_fetch_xml(attributes):
    tags = "\n".join(
        f'<attribute name="{c}"/>' for c in attributes
    )

    return f"""
<fetch version="1.0" output-format="xml-platform" mapping="logical">
  <entity name="cad_inscricaocandidato">
    {tags}
    <order attribute="createdon" descending="true"/>
  </entity>
</fetch>
"""


def post_retry(session, *args, retries=3, **kwargs):
    for i in range(retries):
        try:
            r = session.post(*args, **kwargs)
            r.raise_for_status()
            return r
        except requests.RequestException:
            if i == retries - 1:
                raise
            time.sleep(5)


def get_page(session, page, headers, body):
    headers = headers.copy()
    headers["Pagina"] = str(page)

    response = post_retry(
        session,
        URL,
        headers=headers,
        data=body.encode("utf-8"),
        timeout=60,
    )

    doc = xmltodict.parse(response.text)

    resultset = doc["resultset"]
    records = resultset["result"]

    if isinstance(records, dict):
        records = [records]

    return records, resultset["@morerecords"] == "1"


def get_all_records(session, headers, body):
    page = 1
    rows = []

    while True:
        records, more = get_page(session, page, headers, body)
        rows.extend(records)
        print(f"Página {page}: {len(records)} registros")

        if not more:
            break

        page += 1

    return rows


def flatten_value(value):
    if not isinstance(value, dict):
        return value

    if "@name" in value:
        return value["@name"]

    if "#text" in value:
        return value["#text"]

    return value


def flatten_record(record):
    return {
        k: v['#text'] if k == 'cad_clientepotencialid' else  flatten_value(v)
        for k, v in record.items()
    }


def load_inscricoes():

    with requests.Session() as session:

        token = get_token(session)
        headers = create_headers(token)
        body = create_fetch_xml(atributos)

        records = get_all_records(
            session,
            headers,
            body,
        )
    
    df = pd.DataFrame(
        flatten_record(r)
        for r in records
    )

    for col in ["'createdon'", "modifiedon"]:
        if col in df.columns:
            df[col] = pd.to_datetime(
                df[col],
                format="%d/%m/%Y %H:%M",
                errors="coerce",
            )

    df.rename(
        columns={
            k: v
            for k, v in rename_columns.items()
            if k in df.columns
        },
        inplace=True,
    )

    map = {
        "Faculdade Serra Dourada de " : "",
        "Faculdade Alis" : "Itabirito",
        " dos Carajás" : ""}
    
    df['Unidade Inscrito'] = df['Unidade Inscrito'].map(map)
    return df

inscricoes = load_inscricoes()





Página 1: 250 registros
Página 2: 250 registros
Página 3: 250 registros
Página 4: 250 registros
Página 5: 250 registros
Página 6: 250 registros
Página 7: 250 registros
Página 8: 250 registros
Página 9: 250 registros
Página 10: 250 registros
Página 11: 250 registros
Página 12: 250 registros
Página 13: 250 registros
Página 14: 250 registros
Página 15: 250 registros
Página 16: 250 registros
Página 17: 250 registros
Página 18: 250 registros
Página 19: 250 registros
Página 20: 250 registros
Página 21: 250 registros
Página 22: 250 registros
Página 23: 250 registros
Página 24: 134 registros


In [5]:
print(f'Total de leads: {leads.shape[0]}')
print(f'Total de inscricoes: {inscricoes.shape[0]}')

Total de leads: 14641
Total de inscricoes: 5884


In [6]:
leads = leads[['ID Lead', 'Nome Completo']]

In [7]:
inscricoes = inscricoes[['ID Lead', 'id_inscricao']]

In [8]:
df = leads.merge(inscricoes, how= 'left', on = 'ID Lead')

In [9]:
df.head()

,ID Lead,Nome Completo,id_inscricao
0,{ED821602-28BA-F111-AAAF-6045BD39BC63},Arthur Teste,{ADB4A521-28BA-F111-AAAF-6045BD39BC63}
1,{75470743-A692-4F46-A6CC-0ED7E1258684},Arthur Teste,NaN
2,{AECD1848-307D-4775-9A74-22296704DD49},Arthur Teste,NaN
3,{733AE44B-9A45-445A-B369-05075D47AA6D},Arthur Teste,{CBC0A183-232D-4562-8C2F-356BE88A4706}
4,{F23E75FD-0706-4630-9D32-401D06C3E031},Arthur Teste,NaN


In [10]:
colunas = ['ID Lead','id_inscricao']

for coluna in colunas:
    df[coluna] = df[coluna].str.strip().str.lower()
    df[coluna] = df[coluna].str.replace('{', '').str.replace('}', '')

In [11]:
df['Nome Completo'] = df['Nome Completo'].str.strip().str.lower()

In [12]:
testes = df[df['Nome Completo'].str.contains('teste')]

In [13]:
testes.shape

(35, 3)

In [14]:
testes

,ID Lead,Nome Completo,id_inscricao
0,ed821602-28ba-f111-aaaf-6045bd39bc63,arthur teste,adb4a521-28ba-f111-aaaf-6045bd39bc63
1,75470743-a692-4f46-a6cc-0ed7e1258684,arthur teste,NaN
2,aecd1848-307d-4775-9a74-22296704dd49,arthur teste,NaN
3,733ae44b-9a45-445a-b369-05075d47aa6d,arthur teste,cbc0a183-232d-4562-8c2f-356be88a4706
4,f23e75fd-0706-4630-9d32-401d06c3e031,arthur teste,NaN
5,229ce39a-6ded-4a42-83ca-4e1e3199347e,arthur teste,NaN
6,fb1c2f29-8af8-4484-bd08-c2290fdd0b02,arthur teste,NaN
7,c5ed8a54-edd1-49f9-b93d-c3b156b6dfcd,arthur teste,NaN
60,4c17fb54-4387-424c-befd-116174126c4c,hugo teste,3a5948c9-9586-499c-a019-2bdd08ba3dec
61,cfd10bff-d972-44fa-873a-14c956617867,arthur teste,51ef9902-b862-4ca8-a6fb-57dc9cf8508e


In [15]:
testes.to_csv('dataset.csv', index= False)